In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from icu_sleep_breathing_2020_help_functions import * 

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300

font = {'family' : 'normal',
        'weight' : 'normal',
        'size'   : 7}

matplotlib.rc('font', **font)

### load summary tables:

In [2]:
plots_savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/plots'

In [3]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria(sleeplab_matched=True)

# of subjects ICU before inclusion criteria: 103
# of 12-hour segments ICU before inclusion criteria: 621
# of subjects ICU after inclusion criteria: 103
# of 12-hour segments ICU after inclusion criteria: 621
24-hour segments: 256
12-hour segments: 365

# of subjects sleeplab before inclusion criteria: 307
# of 12-hour segments sleeplab before inclusion criteria: 307
# of subjects sleeplab after inclusion criteria: 307
# of 12-hour segments sleeplab after before inclusion criteria: 307


In [4]:
for modality in ['breathing', 'ecg_nn']:
    var_n2 = 'stages_distribution_MODALITY_N2'.replace('MODALITY', modality)
    var_n3 = 'stages_distribution_MODALITY_N3'.replace('MODALITY', modality)
    var_sum = 'stages_distribution_MODALITY_N2N3'.replace('MODALITY', modality)
    
    summary_days_icu[var_sum] = summary_days_icu[var_n2] + summary_days_icu[var_n3]
    summary_days_sleeplab[var_sum] = summary_days_sleeplab[var_n2] + summary_days_sleeplab[var_n3]


In [5]:
summary_days_icu.loc[:, ['stages_distribution' in x for x in summary_days_icu.columns]] *= 100
summary_days_sleeplab.loc[:, ['stages_distribution' in x for x in summary_days_sleeplab.columns]] *= 100

summary_f_icu = summary_days_icu.loc[summary_days_icu.day_cat == 'f', :]
summary_dn_icu = summary_days_icu.loc[summary_days_icu.day_cat != 'f', :]

summary_days_sleeplab_full = summary_days_sleeplab.copy()

In [6]:
def sleep_summary_barplots(ax, data, variables, x_labels,  color='navy', title = ''):
    
    num = len(variables)
    ax.bar(np.arange(num), data[variables].mean(axis=0), yerr=[np.zeros(num,), data[variables].std(axis=0)], color=color)
    ax.set_xticks(np.arange(num))
    ax.set_xticklabels(x_labels, rotation=90)
    ax.set_title(title)

    for iV in range(num):
        var_mean = int(np.round(data[variables[iV]].mean(),0))
        var_std = int(np.round(data[variables[iV]].std(),0))
        if var_mean > 7:
            ax.annotate(str(var_mean), xy=(iV, 2), color='white', ha='center')
        else:
            ax.annotate(str(var_mean), xy=(iV, var_mean+var_std+2), color='black', ha='center')
    return ax

In [7]:
ahi_categories = ['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_30_100', 'ahi_15_100']
ahi_names = {
    'all': 'All',
    'ahi_0_5': 'AHI < 5',
    'ahi_5_15': '5 < AHI <=15',
    'ahi_15_30': '15 < AHI <= 30',
    'ahi_30_100': 'AHI > 30',
    'ahi_15_100': 'AHI > 15',
}


In [8]:
### 3x2 plot
# cols: data type, rows: stage version (resp, ecg, expert)

variables_template = ['sleep_hours_MODALITY',
             'stages_distribution_MODALITY_S',
             'stages_distribution_MODALITY_R',
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2',
             'stages_distribution_MODALITY_N3',
             'sleep_MODALITY_sfi',
             'sleep_MODALITY_sfi_w',
             'sleep_MODALITY_arousali',
            ]
x_labels = ['Sleep Dur. (h)', 'Sleep Efficiency', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'SFI', 'SFI Wake', 'Arousal I.']

### Sleep Stage and Indices Comparison for Signal Modalities; ICU and all matched sleeplab subjects.

In [9]:
ahi_category = 'all'
summary_days_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_' + ahi_category] == 1]

min_hours_sleep_icu = 4
summary_dn_icu_sel = summary_dn_icu.loc[np.any([summary_dn_icu.sleep_hours_breathing >= min_hours_sleep_icu, 
                                               summary_dn_icu.sleep_hours_ecg_nn >= min_hours_sleep_icu],
                                               axis=0), :]

fig, ax = plt.subplots(3,2,figsize=(6,9))
for ax_tmp in ax.flatten():
    ax_tmp.spines['top'].set_visible(False)
    ax_tmp.spines['right'].set_visible(False)
    ax_tmp.spines['bottom'].set_visible(False)
    ax_tmp.spines['left'].set_visible(False)
    ax_tmp.set_yticks([])
    
# row 1: RESPIRATION COMPARISON.
modality = 'breathing'
variables = [x.replace('MODALITY', modality) for x in variables_template]
title = f'ICU (n={summary_dn_icu_sel.shape[0]}), Respiration'
sleep_summary_barplots(ax[0,0], summary_dn_icu_sel, variables, x_labels,  color='navy', title=title)

modality = 'breathing'
variables = [x.replace('MODALITY', modality) for x in variables_template]
title = f'Sleeplab (n={summary_days_sleeplab.shape[0]}), Respiration'
sleep_summary_barplots(ax[0,1], summary_days_sleeplab, variables, x_labels,  color='cornflowerblue', title=title)

# row 2: ECG COMPARISON.
modality = 'ecg_nn'
variables = [x.replace('MODALITY', modality) for x in variables_template]
title = f'ICU (n={summary_dn_icu_sel.shape[0]}), ECG'
sleep_summary_barplots(ax[1,0], summary_dn_icu_sel, variables, x_labels,  color='darkred', title=title)

modality = 'ecg_nn'
variables = [x.replace('MODALITY', modality) for x in variables_template]
title = f'Sleeplab (n={summary_days_sleeplab.shape[0]}), ECG'
sleep_summary_barplots(ax[1,1], summary_days_sleeplab, variables, x_labels,  color='tomato', title=title)

# row 3: EXPERT, only available for sleeplab
modality = 'expert'
variables = [x.replace('MODALITY', modality) for x in variables_template]
title = f'Sleeplab (n={summary_days_sleeplab.shape[0]}), Expert'
sleep_summary_barplots(ax[2,1], summary_days_sleeplab, variables, x_labels,  color='green', title=title)

ax[2,0].set_xticks([])
ax[2,0].text(0.2, 0.5, 'No Expert Labels available for ICU.')


plt.tight_layout()
# plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_category}.jpg'), dpi=500)
# plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_category}.svg'))


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

findfont: Font family ['normal'] not found. Falling back to DejaVu Sans.
findfont: Font family ['normal'] not found. Falling back to DejaVu Sans.


### same as above but loop over all AHI categories:

In [11]:
for ahi_category in ahi_categories:
    if ahi_category == 'all': continue
        
    summary_days_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_' + ahi_category] == 1]

    fig, ax = plt.subplots(3,2,figsize=(6,9))
    for ax_tmp in ax.flatten():
        ax_tmp.spines['top'].set_visible(False)
        ax_tmp.spines['right'].set_visible(False)
        ax_tmp.spines['bottom'].set_visible(False)
        ax_tmp.spines['left'].set_visible(False)
        ax_tmp.set_yticks([])

    # row 1: RESPIRATION COMPARISON.
    modality = 'breathing'
    variables = [x.replace('MODALITY', modality) for x in variables_template]
    title = f'ICU (n={summary_dn_icu.shape[0]}) \nRespiration'
    sleep_summary_barplots(ax[0,0], summary_dn_icu, variables, x_labels,  color='navy', title=title)

    modality = 'breathing'
    variables = [x.replace('MODALITY', modality) for x in variables_template]
    title = f'Sleeplab {ahi_names[ahi_category]} (n={summary_days_sleeplab.shape[0]}) \nRespiration'
    sleep_summary_barplots(ax[0,1], summary_days_sleeplab, variables, x_labels,  color='cornflowerblue', title=title)

    # row 2: ECG COMPARISON.
    modality = 'ecg_nn'
    variables = [x.replace('MODALITY', modality) for x in variables_template]
    title = f'ICU (n={summary_dn_icu.shape[0]}) \nECG'
    sleep_summary_barplots(ax[1,0], summary_dn_icu, variables, x_labels,  color='darkred', title=title)

    modality = 'ecg_nn'
    variables = [x.replace('MODALITY', modality) for x in variables_template]
    title = f'Sleeplab {ahi_names[ahi_category]} (n={summary_days_sleeplab.shape[0]}) \nECG'
    sleep_summary_barplots(ax[1,1], summary_days_sleeplab, variables, x_labels,  color='tomato', title=title)

    # row 3: EXPERT, only available for sleeplab
    modality = 'expert'
    variables = [x.replace('MODALITY', modality) for x in variables_template]
    title = f'Sleeplab {ahi_names[ahi_category]} (n={summary_days_sleeplab.shape[0]}) \nExpert'
    sleep_summary_barplots(ax[2,1], summary_days_sleeplab, variables, x_labels,  color='green', title=title)

    ax[2,0].set_xticks([])
    ax[2,0].text(0.2, 0.5, 'No Expert Labels available for ICU.')


    plt.tight_layout()
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_ahi_{ahi_category}.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_ahi_{ahi_category}.svg'))


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [11]:
ahi_categories

['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_30_100', 'ahi_15_100']

In [30]:
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 7}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

matplotlib.rc('font', **font)

min_hours_sleep_icu = 4
summary_dn_icu_sel = summary_dn_icu.loc[np.any([summary_dn_icu.sleep_hours_breathing >= min_hours_sleep_icu, 
                                               summary_dn_icu.sleep_hours_ecg_nn >= min_hours_sleep_icu],
                                               axis=0), :]

ahi_categories_to_plot = 'all_low_high'

if ahi_categories_to_plot == 'all':
    ahi_categories_sel = ahi_categories[:-1]
elif ahi_categories_to_plot == 'all_low_high':
    ahi_categories_sel = ['all', 'ahi_0_5', 'ahi_15_100']

n_ahi_categories = len(ahi_categories_sel)


fig, ax = plt.subplots(3, 1+n_ahi_categories, figsize=(8,9))

for ax_tmp in ax.flatten():
    ax_tmp.spines['top'].set_visible(False)
    ax_tmp.spines['right'].set_visible(False)
    ax_tmp.spines['bottom'].set_visible(False)
    ax_tmp.spines['left'].set_visible(False)
    ax_tmp.set_yticks([])
    ax_tmp.tick_params(length=0.05)
# ICU first column:

# row 1: RESPIRATION COMPARISON.
modality = 'breathing'
variables = [x.replace('MODALITY', modality) for x in variables_template]
title = None
sleep_summary_barplots(ax[0,0], summary_dn_icu_sel, variables, x_labels,  color='navy', title=title)

# row 2: ECG COMPARISON.
modality = 'ecg_nn'
variables = [x.replace('MODALITY', modality) for x in variables_template]
title = None
sleep_summary_barplots(ax[1,0], summary_dn_icu_sel, variables, x_labels,  color='darkred', title=title)

ax[2,0].set_xticks([])
ax[2,0].text(0.2, 0.5, 'No Expert Labels\navailable for ICU.')

ax[0,0].text(  
0.5, 1.4, f'ICU \n\n(n={summary_dn_icu_sel.shape[0]})', # 1.27
ha='center', va='top', fontdict = font_headers,
transform=ax[0,0].transAxes
)
    
    
for i_ahi_cat, ahi_category in enumerate(ahi_categories_sel):

    summary_days_sleeplab = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_' + ahi_category] == 1]
    
    ax[0,i_ahi_cat+1].text( 
    0.5, 1.2, f'{ahi_names[ahi_category]} \n(n={summary_days_sleeplab.shape[0]})',
    ha='center', va='top', fontdict = font_headers,
    transform=ax[0,i_ahi_cat+1].transAxes
    )
    # row 1: RESPIRATION COMPARISON.

    modality = 'breathing'
    variables = [x.replace('MODALITY', modality) for x in variables_template]
    title = None
    sleep_summary_barplots(ax[0,i_ahi_cat+1], summary_days_sleeplab, variables, x_labels,  color='cornflowerblue', title=title)

    # row 2: ECG COMPARISON.
    modality = 'ecg_nn'
    variables = [x.replace('MODALITY', modality) for x in variables_template]
    sleep_summary_barplots(ax[1,i_ahi_cat+1], summary_days_sleeplab, variables, x_labels,  color='tomato', title=title)

    # row 3: EXPERT, only available for sleeplab
    modality = 'expert'
    variables = [x.replace('MODALITY', modality) for x in variables_template]

    sleep_summary_barplots(ax[2,i_ahi_cat+1], summary_days_sleeplab, variables, x_labels,  color='green', title=title)


ax[0,(n_ahi_categories+1)//2].text( 
0.5, 1.4, f'Sleeplab',
ha='center', va='top', fontdict = font_headers,
transform=ax[0,(n_ahi_categories+1)//2].transAxes
)

ax[0, 0].text( 
-0.1, 0.5, f'Respiration',
ha='center', va='top', fontdict = font_headers, rotation='vertical',
transform=ax[0, 0].transAxes
)

ax[1, 0].text( 
-0.1, 0.5, f'ECG',
ha='center', va='top', fontdict = font_headers, rotation='vertical',
transform=ax[1, 0].transAxes
)

ax[2, 0].text( 
-0.1, 0.5, f'Expert',
ha='center', va='top', fontdict = font_headers, rotation='vertical',
transform=ax[2, 0].transAxes
)


plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.02, hspace=0.5)
plt.tight_layout()

if 0:
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}.svg'))

fig.set_size_inches(8,7)
plt.subplots_adjust(left=None, bottom=0.1, right=None, top=None, wspace=0.02, hspace=0.5)

for ax_tmp in ax[:2,:].flatten():   
    ax_tmp.set_xticklabels(['Dur', 'Eff', 'R', 'N1', 'N2', 'N3', 'SFI', 'SFW', 'AI'])

plt.tight_layout()
plt.subplots_adjust(bottom=0.14)

if 0:
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_shortxlabels.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_shortxlabels.svg'))

fig.set_size_inches(8,4)
plt.tight_layout()
plt.tight_layout()
plt.subplots_adjust(bottom=0.2)
if 0:
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_shortxlabels_8_4.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_shortxlabels_8_4.svg'))

fig.set_size_inches(6.5,4)
plt.tight_layout()
plt.tight_layout()
plt.subplots_adjust(bottom=0.22)
if 1:
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_shortxlabels_8_4.jpg'), dpi=500)
    plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_shortxlabels_8_4.svg'))


# fig.set_size_inches(8,6)
    
# for ax_tmp in ax[:2,:].flatten():
#     ax_tmp.set_xticklabels([])
#     ax_tmp.set_xticks([])
    
# plt.tight_layout()
# plt.subplots_adjust(bottom=0.14)
# if 0:
#     plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_noxlabels.jpg'), dpi=500)
#     plt.savefig(os.path.join(plots_savedir, f'Figure_Sleep_Stage_Analysis_Histograms_Indices_All_minhoursleep_{str(min_hours_sleep_icu)}_ahi_{ahi_categories_to_plot}_noxlabels.svg'))


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [14]:
plt.close('all')

## Box for each Feature plot

In [15]:
### 3x2 plot
# cols: data type, rows: stage version (resp, ecg, expert)

variables_template = ['sleep_hours_MODALITY',
             'stages_distribution_MODALITY_S',
             'stages_distribution_MODALITY_R',
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2',
             'stages_distribution_MODALITY_N3',
             'sleep_MODALITY_sfi',
             'sleep_MODALITY_sfi_w',
             'sleep_MODALITY_arousali',
            ]

x_labels = ['Sleep Dur. (h)', 'Sleep Efficiency', 'Stage R (%)', 'Stage N1 (%)', 'Stage N2 (%)', 'Stage N3 (%)', 'SFI', 'SFI Wake', 'Arousal I.']

In [16]:
variables

['sleep_hours_expert',
 'stages_distribution_expert_S',
 'stages_distribution_expert_R',
 'stages_distribution_expert_N1',
 'stages_distribution_expert_N2',
 'stages_distribution_expert_N3',
 'sleep_expert_sfi',
 'sleep_expert_sfi_w',
 'sleep_expert_arousali']

In [17]:
def set_shared_ylim(ax1, ax2):
    
    ylim_min1, ylim_max1 = ax1.get_ylim()
    ylim_min2, ylim_max2 = ax2.get_ylim()
    ylim_min = np.min([ylim_min1, ylim_min2])
    ylim_min = np.max([ylim_min, 0])
    ylim_max = np.max([ylim_max1, ylim_max2])
    
    ax1.set_ylim(ylim_min, ylim_max)
    ax2.set_ylim(ylim_min, ylim_max)
    
    return ax1, ax2

In [26]:
sleeplab_all = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_all'] == 1]
sleeplab_ahi_0_5 = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_ahi_0_5'] == 1]
sleeplab_ahi_5_15 = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_ahi_5_15'] == 1]
sleeplab_ahi_15_30 = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_ahi_15_30'] == 1]
sleeplab_ahi_30_100 = summary_days_sleeplab_full.loc[summary_days_sleeplab_full['matched_ahi_30_100'] == 1]

min_hours_sleep_icu = 2
icu_2 = summary_dn_icu.loc[np.any([summary_dn_icu.sleep_hours_breathing >= min_hours_sleep_icu, 
                                               summary_dn_icu.sleep_hours_ecg_nn >= min_hours_sleep_icu],
                                               axis=0), :]
min_hours_sleep_icu = 4
icu_4 = summary_dn_icu.loc[np.any([summary_dn_icu.sleep_hours_breathing >= min_hours_sleep_icu, 
                                               summary_dn_icu.sleep_hours_ecg_nn >= min_hours_sleep_icu],
                                               axis=0), :]

# fig, ax = plt.subplots(3,2,figsize=(6,9))
# for ax_tmp in ax.flatten():
#     ax_tmp.spines['top'].set_visible(False)
#     ax_tmp.spines['right'].set_visible(False)
#     ax_tmp.spines['bottom'].set_visible(False)
#     ax_tmp.spines['left'].set_visible(False)
#     ax_tmp.set_yticks([])
    
modality = 'breathing'
variables = [x.replace('MODALITY', modality) for x in variables_template]


var_name = {
  'sleep_hours_ecg_nn' : 'Sleep Duration (h) [ECG]',
 'stages_distribution_ecg_nn_S': 'Sleep Efficiency [ECG]',
 'stages_distribution_ecg_nn_R': 'Stage R (%) [ECG]',
 'stages_distribution_ecg_nn_N1': 'Stage N1 (%) [ECG]',
 'stages_distribution_ecg_nn_N2': 'Stage N2 (%) [ECG]',
 'stages_distribution_ecg_nn_N3': 'Stage N3 (%) [ECG]',
'stages_distribution_ecg_nn_N2N3': 'Stage N2+N3 (%) [ECG]',

 'sleep_ecg_nn_sfi': 'SFI [ECG]',
 'sleep_ecg_nn_sfi_w': 'SFI-Wake [ECG]',
 'sleep_ecg_nn_arousali': 'Arousal Index [ECG]',
'sleep_hours_breathing' : 'Sleep Duration (h) [Respiration]',
 'stages_distribution_breathing_S': 'Sleep Efficiency [Respiration]',
 'stages_distribution_breathing_R': 'Stage R (%) [Respiration]',
 'stages_distribution_breathing_N1': 'Stage N1 (%) [Respiration]',
 'stages_distribution_breathing_N2': 'Stage N2 (%) [Respiration]',
 'stages_distribution_breathing_N3': 'Stage N3 (%) [Respiration]',
    'stages_distribution_breathing_N2N3': 'Stage N2+N3 (%) [Respiration]',

 'sleep_breathing_sfi': 'SFI [Respiration]',
 'sleep_breathing_sfi_w': 'SFI-Wake [Respiration]',
 'sleep_breathing_arousali': 'Arousal Index [Respiration]',
}

def plot_mean_std_element(data, var, ax, position=0, color='blue', marker=None):
    
    mean_var = data[var].mean()
    std_var = data[var].std()
    ax.errorbar(position, mean_var, yerr=std_var, color=color, fmt='.', zorder=1, marker=marker)
    
    return ax

def plot_mean_std_all_elements(var, ax):
    
    ax = plot_mean_std_element(icu_2, var, ax, position=0, color='royalblue')
    ax = plot_mean_std_element(icu_4, var, ax, position=1, color='navy')
    ax = plot_mean_std_element(sleeplab_all, var, ax, position=2, color='gray')
    ax = plot_mean_std_element(sleeplab_ahi_0_5, var, ax, position=3, color='limegreen')
    ax = plot_mean_std_element(sleeplab_ahi_5_15, var, ax, position=4, color='darkgreen')
    ax = plot_mean_std_element(sleeplab_ahi_15_30, var, ax, position=5, color='orange')
    ax = plot_mean_std_element(sleeplab_ahi_30_100, var, ax, position=6, color='red')

    return ax

variables_to_use = 'n2n3_sum'
    
if variables_to_use == 'all':
    variables_template = ['sleep_hours_MODALITY',
             'stages_distribution_MODALITY_S',
             'stages_distribution_MODALITY_R',
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2',
             'stages_distribution_MODALITY_N3',
             'sleep_MODALITY_sfi',
#              'sleep_MODALITY_sfi_w',
             'sleep_MODALITY_arousali',
            ]
elif variables_to_use == 'n2n3_sum':
    variables_template = ['sleep_hours_MODALITY',
             'stages_distribution_MODALITY_S',
             'stages_distribution_MODALITY_R',
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2N3',
             'sleep_MODALITY_sfi',
             'sleep_MODALITY_arousali',
            ]
        
n_subplots = len(variables_template)*2
fig, ax = plt.subplots(int(np.ceil(n_subplots/4)), 4, figsize=(8,8))

for i_variable, var_temp in enumerate(variables_template):
    
    for i_modality, modality in enumerate(['breathing', 'ecg_nn']):
        
        var = var_temp.replace('MODALITY', modality)
        ax[i_variable // 2, i_modality + 2*(i_variable % 2)] = plot_mean_std_all_elements(var, ax[i_variable // 2, i_modality + 2*(i_variable % 2)])
        ax[i_variable // 2, i_modality + 2*(i_variable % 2)].set_xticks(np.arange(7))
        ax[i_variable // 2, i_modality + 2*(i_variable % 2)].set_xticklabels(['>2h', '>4h', 'All', 'Norm', 'Mild', 'Mod', 'Sev'])
        ax[i_variable // 2, i_modality + 2*(i_variable % 2)].set_title(var_name[var])

    ax[i_variable // 2, i_modality + 2*(i_variable % 2) - 1], ax[i_variable // 2, i_modality + 2*(i_variable % 2)] = set_shared_ylim(ax[i_variable // 2, i_modality + 2*(i_variable % 2) - 1], 
                                  ax[i_variable // 2, i_modality + 2*(i_variable % 2)])
    
# for row in range(int(np.ceil(n_subplots/4))):
#     for axis_each_row in ax[row,:]:
#         axis_each_row.get_shared_y_axes().join(axis_each_row, ax[row, 0])

for axis_bottom in ax[-1, :]:
    axis_bottom.text(0.12, -0.15, f'ICU', ha='center', va='top', rotation='horizontal',
                     fontdict = font_headers, transform=axis_bottom.transAxes)
    axis_bottom.text(0.7, -0.15, f'Sleeplab (AHI category)', ha='center', va='top', rotation='horizontal',
                    fontdict = font_headers, transform=axis_bottom.transAxes)

plt.subplots_adjust(left=None, bottom=0.1, right=None, top=None, wspace=0.02, hspace=0.5)
plt.tight_layout()

plt.savefig(os.path.join(plots_savedir, f'Figure_SleepStagesIndices_Featuregrid_{variables_to_use}.jpg'), dpi=500)
plt.savefig(os.path.join(plots_savedir, f'Figure_SleepStagesIndices_Featuregrid_{variables_to_use}.svg'))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
# Main Sleeplab vs ICU.
# R vs. N1 vs. N2+N3
# both modalities.

In [131]:
font = {'family' : 'normal',
        'weight' : 'normal',
        'size'   : 7}

matplotlib.rc('font', **font)

variables_to_use = 'n2n3_sum'
    
if variables_to_use == 'n2n3_sum':
    variables_template = [
             'stages_distribution_MODALITY_N1',
            'stages_distribution_MODALITY_N2N3',
        'stages_distribution_MODALITY_R',
        'sleep_hours_MODALITY',
             'sleep_MODALITY_sfi',
             'sleep_MODALITY_arousali',
            ]
    var_names = ['N1', 'N2 + N3', 'R', 'Sleep Dur (h)', 'SFI', 'Arousal I']

marker = {'breathing': 'o',
         'ecg_nn': 's'}

legend_handles = []

figdummy, axdummy = plt.subplots(1,1);
plt.close()

fig, ax = plt.subplots(1, 2, figsize=(5,2))

i_pos = 0

for i_var, var_temp in enumerate(variables_template[:3]):

    for modality in ['breathing', 'ecg_nn']:
        var = var_temp.replace('MODALITY', modality)
        ax[0] = plot_mean_std_element(icu_4, var, ax[0], position=i_pos, color='navy', marker=marker[modality])
        ax[0] = plot_mean_std_element(sleeplab_all, var, ax[0], position=i_pos+2, color='orange', marker=marker[modality])
        i_pos += 1
        
#     ax[0].text(i_pos-0.5, 89, var_names[i_var], ha='center', va='top', rotation='horizontal',
#                     fontdict = font_headers)
    i_pos += 3

ax[0].set_xticks([1.5, 6.5, 11.5])
ax[0].set_xticklabels(var_names[:3])


legend_handles.append(axdummy.errorbar(0, 0, yerr=1, color='navy', fmt='.', zorder=1, marker=marker['breathing']))
legend_handles.append(axdummy.errorbar(0, 0, yerr=1, color='navy', fmt='.', zorder=1, marker=marker['ecg_nn']))
legend_handles.append(axdummy.errorbar(0, 0, yerr=1, color='orange', fmt='.', zorder=1, marker=marker['breathing']))
legend_handles.append(axdummy.errorbar(0, 0, yerr=1, color='orange', fmt='.', zorder=1, marker=marker['ecg_nn']))

ax[0].set_ylim([0, 86])
ax[1].legend(legend_handles, ['ICU, Respiration', 'ICU, ECG', 'Sleeplab, Respiration', 'Sleeplab, ECG'], 
             frameon=False, markerscale=1, ncol=1, bbox_to_anchor=(0.2, 0.48, 0.5, 0.5), loc='best')
ax[0].set_ylabel('%')

# Subplot 2:

i_pos = 0

for i_var, var_temp in enumerate(variables_template[3:]):

    for modality in ['breathing', 'ecg_nn']:
        var = var_temp.replace('MODALITY', modality)
        ax[1] = plot_mean_std_element(icu_4, var, ax[1], position=i_pos, color='navy', marker=marker[modality])
        ax[1] = plot_mean_std_element(sleeplab_all, var, ax[1], position=i_pos+2, color='orange', marker=marker[modality])
        i_pos += 1
    i_pos += 3

ax[1].set_xticks([1.5, 6.5, 11.5])
ax[1].set_xticklabels(var_names[3:])
ax[1].set_ylim([0, 26])
# ax[1].set_ylabel('%')



plt.subplots_adjust(left=None, bottom=0.1, right=None, top=None, wspace=0.02, hspace=0.5)
plt.tight_layout()

plt.savefig(os.path.join(plots_savedir, f'Figure_SleepStagesIndices_MainSleeplabVsICU_{variables_to_use}.jpg'), dpi=500)
plt.savefig(os.path.join(plots_savedir, f'Figure_SleepStagesIndices_MainSleeplabVsICU_{variables_to_use}.svg'))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [121]:
plt.close('all')

In [191]:
i_variable % 2

0

In [193]:
i_variable

8

In [128]:
summary_dn_icu.columns[80:120]

Index(['hco3_arterial_max', 'pco2_arterial_mean', 'pco2_arterial_min',
       'pco2_arterial_max', 'po2_arterial_mean', 'po2_arterial_min',
       'po2_arterial_max', 'ph_arterial_mean', 'ph_arterial_min',
       'ph_arterial_max', 'oxygen_flow_rate_max', 'sofa_score_mean',
       'sofa_score_min', 'sofa_score_max', 'ahi_va_a3', 'ahi_va_a3_ss',
       'ahi_vb_a3', 'ahi_vb_a3_ss', 'ahi_ro_a3', 'ahi_ro_a3_ss', 'ahi_rsr_a3',
       'ahi_rsr_a3_ss', 'ahi_expert', 'hypoxic_burden_va_a3',
       'hypoxic_burden_va_a3_ss', 'hypoxic_burden_vb_a3',
       'hypoxic_burden_vb_a3_ss', 'hypoxic_burden_ro_a3',
       'hypoxic_burden_ro_a3_ss', 'hypoxic_burden_rsr_a3',
       'hypoxic_burden_rsr_a3_ss', 'hypoxic_burden_expert',
       'self_similarity_sleep', 'self_similarity_sleep_q50',
       'self_similarity_sleep_q75', 'self_similarity_apnea',
       'self_similarity_apnea_q50', 'self_similarity_apnea_q75', 'opioids_sum',
       'benzos_sum'],
      dtype='object')